# BuoyNet 591 Final Project Pipeline
Run this Google Colab Notebook with GPU Acceleration enabled.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/ColabNotebooks/Codesign/

# Unzip the dataset into the BuoyNet/data/raw directory
import os
os.makedirs('BuoyNet/data/raw', exist_ok=True)

!unzip -n -q "dataset.zip" -d BuoyNet/data/raw/

%cd BuoyNet


In [ ]:
!pip install torch torchvision torchaudio numpy pandas matplotlib opencv-python albumentations scikit-learn nbconvert

## 1. Data Preparation
This step simulates generating/cleaning the train/test splits. You should insert the real dataloading code inside `scripts/prepare_dataset.py`

In [ ]:
!python scripts/prepare_dataset.py

## 2. Baseline Model Training
Trains the baseline FP32 MobileNetV3-Small on the training set.

In [ ]:
!python scripts/train_baseline.py

## 3. Quantization-Aware Training (QAT)
Takes 1-2 hours on Colab A100 GPU

In [ ]:
# Run 15 epochs
!jupyter nbconvert --to script notebooks/BuoyNet_QAT_Pipeline.ipynb
!python notebooks/BuoyNet_QAT_Pipeline.py

## 4. Compression Pipeline
Applies INT8 Post-Training Quantization (naive), Structured L1 Channel Pruning (30/50/70%), and Huffman Encoding. Evaluates theoretical storage boundaries.

In [ ]:
!python scripts/compression_pipeline.py

## 5. Evaluation and Profiling Suite
Augments the test dataset with Turbidity and Fouling. Then, simulates Pi latency scaling based on the Colab execution environment

In [ ]:
!python scripts/augment_domain_shift.py
!python scripts/simulate_latency.py
!python scripts/evaluate_domain_shift.py
!python scripts/ieee_master_eval.py

## 6. Produce Final Visualizations & Metrics
Generates Pareto Accuracy vs Latency Frontier, Confusion Matrix, and metrics CSVs

In [ ]:
!python scripts/analyze_pareto.py
!python scripts/advanced_ieee_plots.py

from IPython.display import Image, display
import pandas as pd

print("E2E Pi Estimated Latency Metrics:")
display(pd.read_csv('figures/pi_latency_estimates.csv'))

print("\nHuffman Coding Ratio Stats:")
display(pd.read_csv('logs/huffman_compression_stats.csv'))

print("\nPareto Accuracy vs Latency Plot")
display(Image('figures/pareto_accuracy_vs_latency.png'))

print("\nClass Degradation Comparison")
display(Image('figures/class_degradation_comparison.png'))